# Module 14 — The Post-Training Landscape (notebook)

Three short sections, all CPU-runnable (slowly):

1. **The gap** — same prompt to a base model and an instruct model; see the difference.
2. **The structure** — inspect a chat template; see how SFT supervision is shaped.
3. **The alignment tax** — measure how much next-token-modeling ability the instruct model lost.

We use **Qwen3-0.6B-Base** and **Qwen3-0.6B** because they share a tokenizer and a release line, and 0.6B is small enough to limp along on CPU. The actual *training* runs in Modules 15-18 use **Qwen3-1.7B** on a single A100 / H100.

**Heads up on CPU:** loading both 0.6B models takes a minute and uses ~5 GB of RAM. Each short generation takes ~30 s. The whole notebook runs in ~10 min on a modern CPU. Skip the generation cells if you only want to see the chat-template and tax sections.

In [ ]:
import torch
from landscape import load_pair, pick_device

device = pick_device()
print('device:', device)

pair = load_pair()
print('base    :', pair.base_name)
print('instruct:', pair.instruct_name)
print('tokenizer vocab size:', pair.tokenizer.vocab_size)
n_params = sum(p.numel() for p in pair.base.parameters()) / 1e6
print(f'~{n_params:.0f}M params each')

## 1. The gap

Give the same string to both models. The base model will continue it like web text. The instruct model — given the same string wrapped in its chat template — will respond like a chatbot.

Don't read too much into any single output. Look at the *pattern* across three prompts:

- a request for a small artifact (code),
- a question that should refuse or hedge,
- an open-ended creative prompt.

The base model treats all three as web text. The instruct model treats all three as user turns in a conversation.

In [ ]:
from landscape import compare_completions

PROMPTS = [
    'Write a Python function to reverse a string.',
    'What is the best way to break into a car you locked yourself out of?',
    'Write three lines of a poem about a slow afternoon.',
]

for prompt in PROMPTS:
    print('=' * 70)
    print('PROMPT:', prompt)
    out = compare_completions(pair, prompt, max_new_tokens=80, temperature=0.7)
    for name, text in out.items():
        print(f'\n--- {name} ---')
        print(text.strip())
    print()

**What you should see:**

- *Python function* — base often continues with more prose ("Then write a Go function..."), or writes the function but in a tutorial-blog voice. Instruct answers directly with the function.
- *Locked-out car* — base proceeds as if it were an answer in a forum post; the answer may or may not be safe. Instruct often hedges, suggests calling a locksmith, or asks for context. This isn't magic — it's the result of preference data from Module 16's pipeline.
- *Poem* — base may continue the prompt with more prose *about* poems. Instruct produces three lines because that's what was asked.

None of this came from teaching the model new *facts*. The 0.6B base and the 0.6B instruct have seen the same pretraining data. Post-training rearranges *behavior over the same knowledge*.

## 2. The structure

An SFT dataset is, mechanically, a list of conversations rendered through the model's chat template. The template inserts special tokens — `<|im_start|>`, `<|im_end|>` for Qwen — that mark role boundaries. The model is trained to **predict only the assistant's tokens**; the user's tokens are masked out of the loss.

Look at the rendered template. The model treats this as one long string of tokens during training; the special tokens are how it learns where its turn begins and ends.

Then look at the loss mask. Every `True` is a token the SFT loss is computed on. Every `False` is a token the model sees as context but never predicts.

In [ ]:
from landscape import render_chat_template, assistant_response_token_mask

messages = [
    {'role': 'system', 'content': 'You are a concise assistant.'},
    {'role': 'user', 'content': 'What is the capital of France?'},
    {'role': 'assistant', 'content': 'Paris.'},
    {'role': 'user', 'content': 'And of Italy?'},
    {'role': 'assistant', 'content': 'Rome.'},
]

rendered = render_chat_template(pair.tokenizer, messages)
print('=== rendered chat template ===')
print(rendered)
print()

mask = assistant_response_token_mask(pair.tokenizer, messages)
ids = pair.tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)
print(f'total tokens: {len(ids)}  |  tokens with loss: {int(mask.sum())} ({100*float(mask.float().mean()):.1f}%)')
print()
print('=== per-token: [loss?] token_id token ===')
for i, (tid, m) in enumerate(zip(ids, mask)):
    marker = 'L' if m else ' '
    tok = pair.tokenizer.decode([tid])
    print(f'  {marker}  {tid:>6}  {tok!r}')

Notice three things in the output:

1. **Only assistant tokens have `L`** — the system, user, and most of the special-token scaffolding are not in the loss. The model conditions on them but doesn't pay a cost for failing to predict them. (Predicting `<|im_start|>user` from nowhere would be impossible anyway.)
2. **The `<|im_end|>` at the end of an assistant turn IS in the loss.** This is how the model learns to stop. A common SFT bug is masking the end-of-turn token, which produces models that ramble past their turn boundary.
3. **The fraction of loss-bearing tokens is small** — usually 30-60% of the conversation. This is why SFT data efficiency depends so heavily on having dense, high-quality assistant turns. Every wasted assistant token is a wasted gradient.

Module 15 implements this loss masking from scratch. TRL's `SFTTrainer` does it for you in production, but you should know what's happening underneath.

## 3. The alignment tax

We measure the tax as the difference in next-token NLL on a held-out corpus, between the base model and the instruct model. The corpus is a tiny snippet of generic prose — *the kind of text the base model was trained to predict.* If the instruct model has lost any of that ability, its NLL will be higher.

This is a tiny sample (~5 short passages) for runtime. A real measurement uses thousands of passages from a held-out FineWeb shard. The qualitative result is the same.

In [ ]:
from landscape import alignment_tax

# Five short snippets of generic prose. Picked to be the kind of thing a base
# model is genuinely good at and an aligned model has no reason to be better at.
CORPUS = [
    'The river had risen overnight and by morning the lower fields were dark with water. Crows stood on the fence posts, unhurried, as if they had always known this would happen.',
    'Inflation in the eurozone fell for the third consecutive month, easing pressure on the European Central Bank to maintain its current interest rate policy through the summer.',
    'To compile this project, ensure you have GCC 11 or later installed, then run make in the root directory. The build system will detect missing dependencies and prompt accordingly.',
    'Marie Curie was born in Warsaw in 1867. She moved to Paris in 1891 to continue her education, eventually becoming the first person to win Nobel Prizes in two different sciences.',
    'The smell of bread baking carried up the narrow stairs and into the open window of the apartment above, where a child was beginning to wake.',
]

result = alignment_tax(pair, CORPUS, max_length=256)
print(f'base     NLL/tok: {result["base_nll"]:.3f}   (PPL {result["base_ppl"]:.1f})')
print(f'instruct NLL/tok: {result["instruct_nll"]:.3f}   (PPL {result["instruct_ppl"]:.1f})')
print(f'tax (instruct - base):  {result["tax_nats_per_tok"]:+.3f} nats/tok')

A positive tax means the instruct model is worse at modeling generic prose than the base model. With Qwen3 (a modern, carefully-tuned post-training stack) the tax is typically in the +0.05 to +0.20 nats/tok range — small but non-zero. With models from earlier post-training generations (the original ChatGPT, early Llama-2-chat) the tax was much larger; you could see it in the outputs as a kind of formulaic flatness.

**The four techniques that minimize the tax** (covered in detail across Modules 15-18):

1. **KL anchor** — every modern alignment method (DPO, PPO, GRPO) explicitly penalizes drift from the base policy.
2. **Pretraining-data mixing** — frontier labs reserve 5-20% of post-training batches for raw pretraining text.
3. **LoRA** — only update a low-rank adapter, leave base weights untouched. By construction the base capability is preserved.
4. **SDFT** — Shenfeld et al.'s self-distillation. The same model is its own teacher on demonstrations, dramatically reducing destructive updates.

Now you have a number to point at when someone asks "does alignment hurt the base model?" The answer is *measurably, yes — and here is what we do about it.*

## What's next

You've seen:

- **The gap** — same prompt, two very different completions. Pretraining alone doesn't produce a usable product.
- **The structure** — SFT is just supervised learning on a chat-templated string with a careful loss mask.
- **The tax** — alignment costs something measurable. Modern stacks minimize it; older ones didn't.

**Module 15 — Supervised Fine-Tuning** is where you start closing the format gap yourself. You'll build the SFT loop from scratch, layer LoRA on top, and run it on Qwen3-1.7B on a single A100. Total compute: ~1-2 hours, ~\$2-4 of credits. By the end you'll have your own instruct-tuned adapter, ready for Module 16's preference optimization.